In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'openai-whisper', 'scipy'],
    check=True,
)
print('packages installed')


In [ ]:
import gc
from importlib.metadata import version

import torch
import whisper

MODEL_NAME = 'small'

if not torch.cuda.is_available():
    raise RuntimeError('select a GPU runtime before continuing')

# release an earlier model before switching sizes.
if globals().get('loaded_model_name') != MODEL_NAME or 'model' not in globals():
    globals().pop('model', None)
    gc.collect()
    torch.cuda.empty_cache()
    model = whisper.load_model(MODEL_NAME, device='cuda')
    loaded_model_name = MODEL_NAME

print('model:', loaded_model_name)
print('gpu:', torch.cuda.get_device_name(0))
print('python:', sys.version.split()[0])
print('torch:', torch.__version__)
print('openai-whisper:', version('openai-whisper'))
print('scipy:', version('scipy'))


In [ ]:
import math
import threading
import time
import traceback

import numpy as np
from google.colab import output
from IPython.display import JSON
from scipy.signal import resample_poly

model_lock = threading.Lock()

def transcribe_microphone(samples, sample_rate):
    try:
        sample_rate = int(sample_rate)
        audio = np.asarray(samples, dtype=np.float32)
        if (
            not 8000 <= sample_rate <= 192000
            or audio.ndim != 1
            or audio.size == 0
            or audio.size > sample_rate * 8
            or not np.isfinite(audio).all()
        ):
            raise ValueError('invalid microphone audio')

        # whisper needs one channel at 16000 samples per second.
        divisor = math.gcd(sample_rate, 16000)
        audio = resample_poly(
            audio, 16000 // divisor, sample_rate // divisor
        ).astype(np.float32)

        with model_lock, torch.inference_mode():
            torch.cuda.synchronize()
            started = time.perf_counter()
            result = model.transcribe(
                audio, language='en', task='transcribe', fp16=True,
                temperature=0, condition_on_previous_text=False, verbose=None,
            )
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started

        return JSON({
            'ok': True, 'text': result['text'].strip(),
            'seconds': round(elapsed, 2), 'model': loaded_model_name,
        })
    except Exception as error:
        traceback.print_exc()
        return JSON({'ok': False, 'error': f'{type(error).__name__}: {error}'})

output.register_callback('live_whisper.transcribe', transcribe_microphone)
print('transcription function ready — run cell 4')


In [ ]:
from google.colab.output import eval_js

print(eval_js(r"""
(() => {
  const BATCH_SECONDS = 5;
  const MAX_SECONDS = 120;
  const panel = document.createElement('div');
  panel.style.cssText = 'padding:16px;border:1px solid #aaa;border-radius:10px;';
  const start = document.createElement('button');
  const stop = document.createElement('button');
  const status = document.createElement('p');
  const transcript = document.createElement('pre');
  start.textContent = 'Start microphone';
  stop.textContent = 'Stop microphone';
  stop.disabled = true;
  start.style.marginRight = '10px';
  transcript.style.cssText = 'white-space:pre-wrap;font-size:17px;line-height:1.5;';
  status.textContent = 'Ready. Completed text will stay below.';
  panel.append(start, stop, status, transcript);
  document.body.appendChild(panel);

  let recording = false;
  let stream, context, source, recorder, stopTimer;
  let sampleRate = 48000;
  let pieces = [], pieceSamples = 0, totalSamples = 0;
  let captureError = null;
  const queue = [], completed = [];
  const wait = ms => new Promise(resolve => setTimeout(resolve, ms));

  function queueAudio() {
    if (!pieceSamples) return;
    const audio = new Float32Array(pieceSamples);
    let offset = 0;
    for (const piece of pieces) { audio.set(piece, offset); offset += piece.length; }
    queue.push(audio);
    pieces = [];
    pieceSamples = 0;
  }

  function stopRecording() {
    if (!recording) return;
    recording = false;
    clearTimeout(stopTimer);
    stop.disabled = true;
    stream?.getTracks().forEach(track => track.stop());
    source?.disconnect();
    recorder?.disconnect();
    queueAudio();
    status.textContent = 'Microphone stopped. Finishing queued audio…';
  }
  stop.onclick = stopRecording;

  async function transcribe(audio) {
    let timeout;
    try {
      return await Promise.race([
        google.colab.kernel.invokeFunction(
          'live_whisper.transcribe', [Array.from(audio), sampleRate], {}
        ),
        new Promise((_, reject) => {
          timeout = setTimeout(() => reject(new Error(
            'Python took too long to reply. Check cell 3 before restarting.'
          )), 120000);
        })
      ]);
    } finally { clearTimeout(timeout); }
  }

  start.onclick = async () => {
    start.disabled = true;
    let stage = 'microphone permission';
    try {
      stream = await navigator.mediaDevices.getUserMedia({
        audio: { channelCount: 1, echoCancellation: true, noiseSuppression: true }
      });
      stage = 'audio capture setup';
      context = new AudioContext();
      await context.resume();
      sampleRate = context.sampleRate;

      const processorCode = `
        class MicrophoneCapture extends AudioWorkletProcessor {
          process(inputs) {
            const channel = inputs[0]?.[0];
            if (channel) this.port.postMessage(new Float32Array(channel));
            return true;
          }
        }
        registerProcessor('microphone-capture', MicrophoneCapture);
      `;
      const url = URL.createObjectURL(new Blob([processorCode], {type: 'application/javascript'}));
      try { await context.audioWorklet.addModule(url); }
      finally { URL.revokeObjectURL(url); }

      recorder = new AudioWorkletNode(context, 'microphone-capture');
      recorder.onprocessorerror = () => {
        captureError = 'The browser audio recorder stopped unexpectedly.';
        stopRecording();
      };
      recorder.port.onmessage = event => {
        if (!recording) return;
        pieces.push(event.data);
        pieceSamples += event.data.length;
        totalSamples += event.data.length;
        if (pieceSamples >= sampleRate * BATCH_SECONDS) queueAudio();
        if (totalSamples >= sampleRate * MAX_SECONDS || queue.length >= 24) stopRecording();
      };
      source = context.createMediaStreamSource(stream);
      source.connect(recorder);
      // the processor outputs silence rather than playing back the microphone.
      recorder.connect(context.destination);
      recording = true;
      stop.disabled = false;
      stream.getAudioTracks()[0].addEventListener('ended', stopRecording);
      stopTimer = setTimeout(stopRecording, MAX_SECONDS * 1000);
      status.textContent = 'Listening. Text arrives after each ' + BATCH_SECONDS + '-second batch.';

      while (recording || queue.length) {
        if (!queue.length) { await wait(100); continue; }
        stage = 'browser connection and transcription';
        const audio = queue.shift();
        status.textContent = 'Transcribing. Waiting batches: ' + queue.length;
        const reply = await transcribe(audio);
        let result = reply?.data?.['application/json'];
        if (typeof result === 'string') result = JSON.parse(result);
        if (!result) throw new Error('No JSON result. Run cell 3 and inspect its output.');
        if (!result.ok) throw new Error(result.error);
        if (result.text.trim()) completed.push(result.text.trim());
        transcript.textContent = completed.join(' ');
        status.textContent = (recording ? 'Listening. ' : 'Finishing. ') +
          'Model: ' + result.model + '. Last model time: ' + result.seconds +
          ' seconds. Waiting batches: ' + queue.length;
      }
      if (captureError) throw new Error(captureError);
      status.textContent = 'Finished. Text stays below. Rerun cell 4 for another recording.';
    } catch (error) {
      stopRecording();
      status.textContent = 'Failed during ' + stage + ': ' +
        (error?.message || String(error)) + '. Completed text kept; remaining audio discarded.';
    } finally {
      recording = false;
      clearTimeout(stopTimer);
      stop.disabled = true;
      stream?.getTracks().forEach(track => track.stop());
      source?.disconnect();
      recorder?.disconnect();
      if (context && context.state !== 'closed') await context.close();
      pieces = [];
      queue.length = 0;
    }
  };
  return 'microphone controls ready';
})()
"""))
